In [ ]:
from langgraph_sdk import get_client
from dotenv import load_dotenv
import os

load_dotenv()

🔗 Connected to LangGraph server


In [ ]:
# Set LANGSMITH_API_KEY in your environment
client = get_client(
    url="https://cc-lsd-deep-agent-assistant-15de37b3b13051ee9f72140dbca7989a.us.langgraph.app",
    api_key=os.environ["LANGSMITH_API_KEY"],
)
print("🔗 Connected to LangGraph server")

In [11]:
# 2. Create a new assistant configuration
print("🤖 Creating a new assistant configuration...")

assistant = await client.assistants.create(
    graph_id = GRAPH_ID,
    config={
        "configurable": {
            "system_prompt": "You are a helpful AI assistant. Always reply in the voice of a pirate from the 1600s.",
            "model": "anthropic:claude-haiku-4-5",
            "selected_tools": ["get_todays_date", "advanced_research"],
            "name": "Pirate"
        }
    },
    name="Pirate",
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")
print(f"   🧠 Model: {assistant['config']['configurable']['model']}")
print(f"   🛠️  Tools: {', '.join(assistant['config']['configurable']['selected_tools'])}")

🤖 Creating a new assistant configuration...
✅ Assistant created successfully!
   📍 Assistant ID: bc559a4a-2827-4f96-bbd8-e790fc302a21
   📝 Name: Pirate
   🔢 Version: 1
   🧠 Model: anthropic:claude-haiku-4-5
   🛠️  Tools: get_todays_date, advanced_research


In [12]:
# 2. Create a new assistant configuration
print("🤖 Creating a new assistant configuration...")

assistant = await client.assistants.create(
    graph_id = GRAPH_ID,
    config={
        "configurable": {
            "system_prompt": "You are a helpful AI assistant. Always reply in the voice of a cowboy from the Old West.",
            "model": "anthropic:claude-haiku-4-5",
            "selected_tools": ["get_todays_date", "advanced_research", "finance_research"],
            "name": "Cowboy"
        }
    },
    name="Cowboy"
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")
print(f"   🧠 Model: {assistant['config']['configurable']['model']}")
print(f"   🛠️  Tools: {', '.join(assistant['config']['configurable']['selected_tools'])}")

🤖 Creating a new assistant configuration...
✅ Assistant created successfully!
   📍 Assistant ID: cde89bb3-27f0-42e6-a792-a4cc180392db
   📝 Name: Cowboy
   🔢 Version: 1
   🧠 Model: anthropic:claude-haiku-4-5
   🛠️  Tools: get_todays_date, advanced_research, finance_research


In [13]:
# Store AGENTS.md in the LangGraph persistent store for each assistant
# content-writer/AGENTS.md → Pirate Assistant
# mcp-docs-agent/AGENTS.md → Cowboy Assistant
#
# StoreBackend expects value = {"content": list[str], "created_at": str, "modified_at": str}

from datetime import datetime, timezone

now = datetime.now(timezone.utc).isoformat()

# Read AGENTS.md files from disk
with open("../src/assistants/content-writer/AGENTS.md") as f:
    content_writer_content = f.read()

with open("../src/assistants/mcp-docs-agent/AGENTS.md") as f:
    mcp_docs_content = f.read()

# Map assistant name → AGENTS.md content
agents_md_by_name = {
    "Pirate": content_writer_content,
    "Cowboy": mcp_docs_content,
}

all_assistants = await client.assistants.search(graph_id=GRAPH_ID)
print(f"📋 Found {len(all_assistants)} assistant(s) for graph '{GRAPH_ID}'\n")

for asst in all_assistants:
    name = asst["name"]
    if name not in agents_md_by_name:
        continue

    content = agents_md_by_name[name]
    store_value = {
        "content": content.split("\n"),
        "created_at": now,
        "modified_at": now,
    }

    namespace = [asst["assistant_id"]]
    await client.store.put_item(namespace, key="/AGENTS.md", value=store_value)

    item = await client.store.get_item(namespace, key="/AGENTS.md")
    print(f"✅ Stored AGENTS.md for '{name}'")
    print(f"   └─ Source    : src/assistants/{'content-writer' if name == 'Pirate Assistant' else 'mcp-docs-agent'}/AGENTS.md")
    print(f"   └─ Namespace : {namespace}")
    print(f"   └─ Key       : {item['key']}")
    print(f"   └─ Lines     : {len(item['value']['content'])}")
    print(f"   └─ Updated   : {item['updated_at']}\n")


📋 Found 3 assistant(s) for graph 'deep_agent'

✅ Stored AGENTS.md for 'Cowboy'
   └─ Source    : src/assistants/mcp-docs-agent/AGENTS.md
   └─ Namespace : ['cde89bb3-27f0-42e6-a792-a4cc180392db']
   └─ Key       : /AGENTS.md
   └─ Lines     : 40
   └─ Updated   : 2026-04-14T01:45:03.012223+00:00

✅ Stored AGENTS.md for 'Pirate'
   └─ Source    : src/assistants/mcp-docs-agent/AGENTS.md
   └─ Namespace : ['bc559a4a-2827-4f96-bbd8-e790fc302a21']
   └─ Key       : /AGENTS.md
   └─ Lines     : 32
   └─ Updated   : 2026-04-14T01:45:03.152648+00:00



In [14]:
# Stream token-by-token using stream_mode="messages"
thread = await client.threads.create()
print(f"🧵 Thread created: {thread['thread_id']}\n")

input_data = {"messages": [{"role": "human", "content": "What tools do you have available?"}]}

print("💬 Assistant: ", end="", flush=True)

# Track last seen text per message ID — each partial event contains cumulative content, not deltas
last_seen: dict[str, str] = {}

async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant["assistant_id"],
    input=input_data,
    stream_mode="messages",
):
    if chunk.event == "messages/partial":
        for msg in chunk.data:
            if msg.get("type") == "ai":
                msg_id = msg.get("id", "")
                for block in msg.get("content", []):
                    if block.get("type") == "text":
                        full_text = block.get("text", "")
                        prev_text = last_seen.get(msg_id, "")
                        # Only print the newly arrived portion
                        if full_text.startswith(prev_text):
                            print(full_text[len(prev_text):], end="", flush=True)
                        last_seen[msg_id] = full_text

print()  # newline at end


🧵 Thread created: d85c513c-6f78-4661-bd28-eab37e13193e

💬 Assistant: 
